In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5080
VRAM: 15.92 GB


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project:", PROJECT_ROOT)
print("\nProcessed files:")

for file in PROCESSED_DIR.iterdir():
    print(" -", file.name)


Project: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI

Processed files:
 - executive_regulations.json
 - faiss.index
 - labor_law_articles.json
 - main_articles.json
 - rag_chunks.json
 - rag_embeddings.npy
 - workplace_regulation_articles.json


In [3]:
import json
import numpy as np
import faiss

chunks_path = PROCESSED_DIR / "rag_chunks.json"
embeddings_path = PROCESSED_DIR / "rag_embeddings.npy"
index_path = PROCESSED_DIR / "faiss.index"

with open(chunks_path, "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

rag_embeddings = np.load(embeddings_path)
faiss_index = faiss.read_index(str(index_path))

print("Chunks:", len(rag_chunks))
print("Embeddings:", rag_embeddings.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("FAISS dimension:", faiss_index.d)

Chunks: 162
Embeddings: (162, 768)
FAISS vectors: 162
FAISS dimension: 768


In [4]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda"
)

print("Embedding model loaded.")
print("Device:", embedding_model.device)
print("Dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda:0
Dimension: 768


In [5]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    5
)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"\n{'=' * 70}")
    print(f"Rank {rank} | Score: {score:.4f}")
    print(f"Article: {metadata.get('article')}")
    print(f"Type: {metadata.get('article_type')}")
    print(f"Pages: {metadata.get('page_start')}-{metadata.get('page_end')}")
    print(f"Text:\n{chunk['text']}")


Rank 1 | Score: 0.7325
Article: 8
Type: workplace_regulation
Pages: 30-30
Text:
إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل

Rank 2 | Score: 0.6579
Article: 19
Type: executive_regulation
Pages: 15-15
Text:
في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.

Rank 3 | Score: 0.6463
Article: 33
Type: workplace_regulation
Pages: 33-33
Text:
يستحق العامل عن كل سنة من سنوات 

In [6]:
print("Embedding dtype:", rag_embeddings.dtype)
print("First vector norm:", np.linalg.norm(rag_embeddings[0]))
print("FAISS metric:", faiss_index.metric_type)
print("FAISS is normalized:", np.isclose(np.linalg.norm(rag_embeddings[0]), 1.0, atol=1e-3))

Embedding dtype: float32
First vector norm: 1.0
FAISS metric: 0
FAISS is normalized: True


In [7]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

# Embed the query
query_vec = embedding_model.encode(
    query,
    normalize_embeddings=True
)

# Compare it with every chunk directly
direct_scores = rag_embeddings @ query_vec

# Show the 10 most similar chunks
top_indices = np.argsort(direct_scores)[::-1][:10]

for rank, idx in enumerate(top_indices, start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"\n{'=' * 70}")
    print(f"Rank {rank} | Score: {direct_scores[idx]:.4f}")
    print(f"Article: {metadata['article']}")
    print(f"Type: {metadata['article_type']}")
    print(f"Text: {chunk['text'][:500]}")


Rank 1 | Score: 0.7325
Article: 8
Type: workplace_regulation
Text: إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل

Rank 2 | Score: 0.6579
Article: 19
Type: executive_regulation
Text: في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.

Rank 3 | Score: 0.6463
Article: 33
Type: workplace_regulation
Text: يستحق العامل عن كل سنة من سنوات الخدمة إجازة سنوية بأجر كامل لا تقل مدت

In [8]:
text = rag_chunks[0]["text"]

fresh_embedding = embedding_model.encode(
    text,
    normalize_embeddings=True
)

saved_embedding = rag_embeddings[0]

print("Saved norm:", np.linalg.norm(saved_embedding))
print("Fresh norm:", np.linalg.norm(fresh_embedding))

print(
    "Similarity between saved and fresh:",
    np.dot(saved_embedding, fresh_embedding)
)

Saved norm: 1.0
Fresh norm: 1.0
Similarity between saved and fresh: 1.0


In [9]:
# Generate fresh embeddings locally on the RTX 5080

texts = [chunk["text"] for chunk in rag_chunks]

rag_embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", rag_embeddings.shape)
print("First vector norm:", np.linalg.norm(rag_embeddings[0]))

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding shape: (162, 768)
First vector norm: 1.0


In [10]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

query_vec = embedding_model.encode(
    query,
    normalize_embeddings=True
)

fresh_scores = rag_embeddings @ query_vec
top_indices = np.argsort(fresh_scores)[::-1][:5]

for rank, idx in enumerate(top_indices, start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"\n{'=' * 70}")
    print(f"Rank {rank} | Score: {fresh_scores[idx]:.4f}")
    print(f"Article: {metadata['article']}")
    print(f"Type: {metadata['article_type']}")
    print(f"Pages: {metadata['page_start']}-{metadata['page_end']}")
    print(f"Text: {chunk['text'][:700]}")


Rank 1 | Score: 0.7325
Article: 8
Type: workplace_regulation
Pages: 30-30
Text: إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل

Rank 2 | Score: 0.6579
Article: 19
Type: executive_regulation
Pages: 15-15
Text: في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.

Rank 3 | Score: 0.6463
Article: 33
Type: workplace_regulation
Pages: 33-33
Text: يستحق العامل عن كل سنة من سنوات 

In [11]:
np.save(embeddings_path, rag_embeddings)

print("Saved:", embeddings_path)
print("Shape:", rag_embeddings.shape)

Saved: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\rag_embeddings.npy
Shape: (162, 768)


In [12]:
# Rebuild FAISS using the corrected embeddings

faiss_index = faiss.IndexFlatIP(rag_embeddings.shape[1])
faiss_index.add(rag_embeddings)

faiss.write_index(faiss_index, str(index_path))

print("FAISS index rebuilt.")
print("Vectors:", faiss_index.ntotal)
print("Dimension:", faiss_index.d)

FAISS index rebuilt.
Vectors: 162
Dimension: 768


In [13]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

query_vec = embedding_model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(query_vec, 5)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"\n{'=' * 70}")
    print(f"Rank {rank} | Score: {score:.4f}")
    print(f"Article: {metadata['article']}")
    print(f"Type: {metadata['article_type']}")
    print(f"Pages: {metadata['page_start']}-{metadata['page_end']}")
    print(f"Text: {chunk['text'][:700]}")


Rank 1 | Score: 0.7325
Article: 8
Type: workplace_regulation
Pages: 30-30
Text: إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل

Rank 2 | Score: 0.6579
Article: 19
Type: executive_regulation
Pages: 15-15
Text: في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.

Rank 3 | Score: 0.6463
Article: 33
Type: workplace_regulation
Pages: 33-33
Text: يستحق العامل عن كل سنة من سنوات 

In [14]:
import transformers
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

W0908 06:39:44.363000 23208 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Transformers: 5.16.1
Accelerate: 1.14.0
BitsAndBytes: 0.50.2


In [15]:
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-14B-AWQ"

model_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(PROJECT_ROOT / "models" / "Qwen3-14B-AWQ")
)

print("Model path:")
print(model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Model path:
C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\models\Qwen3-14B-AWQ


In [16]:
%pip install gptqmodel

     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
     ---------------------------------------- 1.3/1.3 MB 5.4 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached device_smi-0.5.7.tar.gz (26 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  U

  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [56 lines of output]
  Cloning into 'C:\Users\User\AppData\Local\Temp\pip-install-govsxdyt\pypcre_763b9a8fdb4543cc874990f61c151d69\pcre_ext\pcre2-10.46'...
  Note: switching to 'b2bd4254b379b9d7dc9a3dda060a7e27009ccdff'.
  
  You are in 'detached HEAD' state. You can look around, make experimental
  changes and commit them, and you can discard any commits you make in this
  state without impacting any branches by switching back to a branch.
  
  If you want to create a new branch to retain commits you create, you may
  do so (now or later) by using -c with the switch command. Example:
  
    git switch -c <new-branch-name>
  
  Or undo this operation with:
  
    git switch -
  
  Turn off this advice by setting config variable advice.detachedHead to false
  
  Submodule 'deps/sljit' (https://github.com/zherczeg/sljit.git) registered for path 'deps/sljit'
  Cloni

In [17]:
!git --version

git version 2.55.0.windows.5


In [18]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODEL_PATH = PROJECT_ROOT / "models" / "Qwen3-14B-AWQ"

print("Project:", PROJECT_ROOT)
print("Model folder exists:", MODEL_PATH.exists())

Project: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI
Model folder exists: True


In [19]:
import os
from pathlib import Path

matches = []

for root, dirs, files in os.walk(Path.home()):
    for file in files:
        if file == "Qwen3-14B-Q4_K_M.gguf":
            matches.append(Path(root) / file)

print("Found:")
for path in matches:
    print(path)

Found:
C:\Users\User\.cache\huggingface\hub\models--Qwen--Qwen3-14B-GGUF\snapshots\530227a7d994db8eca5ab5ced2fb692b614357fd\Qwen3-14B-Q4_K_M.gguf
C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\models\Qwen3-14B-GGUF\Qwen3-14B-Q4_K_M.gguf


In [20]:
MODEL_DIR = PROJECT_ROOT / "models" / "Qwen3-14B-GGUF"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(MODEL_DIR)

C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\models\Qwen3-14B-GGUF


In [21]:
import shutil

SOURCE_MODEL = Path(
    r"C:\Users\User\.cache\huggingface\hub\models--Qwen--Qwen3-14B-GGUF"
    r"\snapshots\530227a7d994db8eca5ab5ced2fb692b614357fd"
    r"\Qwen3-14B-Q4_K_M.gguf"
)

DEST_MODEL = MODEL_DIR / "Qwen3-14B-Q4_K_M.gguf"

shutil.copy2(SOURCE_MODEL, DEST_MODEL)

print("Copied successfully:", DEST_MODEL.exists())
print(f"Size: {DEST_MODEL.stat().st_size / (1024**3):.2f} GB")

Copied successfully: True
Size: 8.38 GB


In [22]:
import shutil

llama_path = shutil.which("llama-cli")

print("llama-cli:", llama_path)

llama-cli: C:\Users\User\AppData\Local\Microsoft\WinGet\Packages\ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe\llama-cli.EXE


In [23]:
import sys

print("Python:", sys.executable)

Python: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\.venv\Scripts\python.exe


In [24]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

files = sorted(p.name for p in PROCESSED_DIR.iterdir())

print("Processed files:")
for f in files:
    print(" -", f)

Processed files:
 - executive_regulations.json
 - faiss.index
 - labor_law_articles.json
 - main_articles.json
 - rag_chunks.json
 - rag_embeddings.npy
 - workplace_regulation_articles.json


In [25]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load chunks
with open(PROCESSED_DIR / "rag_chunks.json", "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

# Load embeddings
rag_embeddings = np.load(PROCESSED_DIR / "rag_embeddings.npy")

# Load FAISS index
faiss_index = faiss.read_index(str(PROCESSED_DIR / "faiss.index"))

print("Chunks:", len(rag_chunks))
print("Embeddings:", rag_embeddings.shape)
print("FAISS vectors:", faiss_index.ntotal)

Chunks: 162
Embeddings: (162, 768)
FAISS vectors: 162


In [26]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device="cuda"
)

print("Embedding model loaded.")
print("Device:", embedding_model.device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda:0


In [27]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    k=3
)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"\nRank {rank}")
    print("Score:", round(float(score), 4))
    print("Article:", metadata["article"])
    print("Type:", metadata["article_type"])
    print("Page:", metadata["page_start"])
    print("Text:", chunk["text"][:300])


Rank 1
Score: 0.7325
Article: 8
Type: workplace_regulation
Page: 30
Text: إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء

Rank 2
Score: 0.6579
Article: 19
Type: executive_regulation
Page: 15
Text: في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة ا

Rank 3
Score: 0.6463
Article: 33
Type: workplace_regulation
Page: 33
Text: يستحق العامل عن كل سنة من سنوات الخدمة إجازة سنوية بأجر كامل لا تقل مدتها عن واحد وعشرين يوماً ، تزاد إلى مدة لا تقل عن ثلاثين
يوماً ، إذا بلغت خدمته خمس سنوات متصلة ، ولل

In [28]:
import requests

response = requests.get("http://127.0.0.1:8080/health")

print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: {"status":"ok"}


In [29]:
import requests

payload = {
    "messages": [
        {
            "role": "user",
            "content": "قل مرحباً بالعربية في جملة واحدة."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 50
}

response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json=payload
)

print("Status:", response.status_code)
print(response.json()["choices"][0]["message"]["content"])

Status: 200
أهلاً وسهلاً! (Ahlan wa sahlan!) - وهذا يعني "أهلاً بك!"


In [30]:
print(response.json())

{'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'role': 'assistant', 'content': 'أهلاً وسهلاً! (Ahlan wa sahlan!) - وهذا يعني "أهلاً بك!"'}}], 'created': 1788838852, 'model': 'C:\\Users\\User\\OneDrive\\Desktop\\Saudi-Compliance-AI\\models\\Gemma3-12B-GGUF\\gemma-3-12b-it-Q4_K_M.gguf', 'system_fingerprint': 'b10730-09412af38', 'object': 'chat.completion', 'usage': {'completion_tokens': 24, 'prompt_tokens': 19, 'total_tokens': 43, 'prompt_tokens_details': {'cached_tokens': 0}}, 'id': 'chatcmpl-mGuT0HnzcAP0bFsT76QGmBbssoB4HpSb', 'timings': {'cache_n': 0, 'prompt_n': 19, 'prompt_ms': 850.521, 'prompt_per_token_ms': 44.76426315789473, 'prompt_per_second': 22.339248531194407, 'predicted_n': 24, 'predicted_ms': 6644.641, 'predicted_per_token_ms': 288.8974347826087, 'predicted_per_second': 3.4614360655451515}}


In [31]:
payload = {
    "messages": [
        {
            "role": "user",
            "content": "قل مرحباً بالعربية في جملة واحدة."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 100,
    "chat_template_kwargs": {
        "enable_thinking": False
    }
}

response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json=payload
)

result = response.json()

print("Status:", response.status_code)
print("Answer:", result["choices"][0]["message"]["content"])
print("Finish reason:", result["choices"][0]["finish_reason"])

Status: 200
Answer: أهلاً وسهلاً! (Ahlan wa sahlan!) - وهذا يعني "مرحباً بك!"
Finish reason: stop


In [32]:
query = "ما هي مدة فترة التجربة في عقد العمل؟"

# Retrieve top 3 relevant chunks
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    k=2
)

# Build the legal context
context_parts = []

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    context_parts.append(
        f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
    )

context = "\n\n".join(context_parts)

print(context)

المصدر 1:
نوع المصدر: workplace_regulation
المادة: 8
الصفحة: 30

النص:
إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل


المصدر 2:
نوع المصدر: executive_regulation
المادة: 19
الصفحة: 15

النص:
في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.



In [33]:
user_prompt = f"""السؤال:
{query}

النص القانوني:
{rag_chunks[indices[0][0]]["text"]}

{rag_chunks[indices[0][1]]["text"]}

أجب عن السؤال باللغة العربية فقط، وبجملة أو جملتين."""

In [34]:
system_prompt = """أنت مساعد قانوني متخصص في نظام العمل السعودي.

أجب باللغة العربية فقط.

اعتمد على السياق القانوني للإجابة، لكن لا تنسخ النص القانوني ولا تعيد صياغة أسماء المواد حرفياً.

أجب بإجابة قصيرة ومباشرة.
لا تستخدم أي لغة غير العربية.
لا تضف معلومات غير موجودة في السياق.
إذا لم تجد الإجابة في السياق، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
"""

In [35]:
test_context = """
تنص المادة الثالثة والخمسون من نظام العمل على ألا يزيد مجموع مدة التجربة في جميع الأحوال على مائة وثمانين يوماً.

وتنص المادة 19 من اللائحة التنفيذية على ألا تتجاوز مدة التجربة 180 يوماً، ويجوز تقسيم مدة التجربة إلى مدد متعددة شريطة أن تكون هذه المدد متصلة وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
"""

user_prompt = f"""السؤال:
ما هي مدة فترة التجربة في عقد العمل؟

السياق:
{test_context}

أجب باللغة العربية فقط وباختصار."""

In [36]:
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.0,
        "max_tokens": 300,
        "chat_template_kwargs": {"enable_thinking": False}
    }
)

result = response.json()

print("Status:", response.status_code)
print("\nAnswer:")
print(result["choices"][0]["message"]["content"])
print("\nFinish reason:", result["choices"][0]["finish_reason"])

Status: 200

Answer:
لا تزيد مدة فترة التجربة في عقد العمل على 180 يومًا.

Finish reason: stop


In [37]:
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": 0.2,
        "max_tokens": 500,
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

result = response.json()

print("Status:", response.status_code)
print("\nAnswer:")
print(result["choices"][0]["message"]["content"])
print("\nFinish reason:", result["choices"][0]["finish_reason"])

Status: 200

Answer:
لا تزيد مدة فترة التجربة في عقد العمل على 180 يومًا.

Finish reason: stop


In [38]:
system_prompt = """أنت مساعد متخصص في نظام العمل السعودي واللوائح التنفيذية المرتبطة به.

قواعد الإجابة:
1. اعتمد فقط على النصوص القانونية الموجودة في السياق المسترجع.
2. أجب مباشرة عن سؤال المستخدم، ولا تضف معلومات غير مطلوبة.
3. لا تخترع أي مادة أو حكم أو معلومة غير موجودة في السياق.
4. حافظ على أرقام وأسماء المواد كما وردت في السياق.
5. اذكر المادة ومصدرها عند الاستشهاد بالحكم.
6. إذا كان السياق المسترجع لا يحتوي على معلومات كافية للإجابة، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
7. أجب باللغة العربية فقط.
8. لا تستخدم أي لغة أخرى أو كلمات أجنبية في الإجابة.
9. لا تكرر النص القانوني حرفياً؛ لخّص الحكم المطلوب باختصار.
"""

user_prompt = f"""السؤال:
{query}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه."""

In [39]:
query = "كم مدة إجازة عيد الفطر للعمال؟"

In [40]:
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 300,
        "chat_template_kwargs": {"enable_thinking": False}
    }
)

result = response.json()

print("Status:", response.status_code)
print("\nAnswer:")
print(result["choices"][0]["message"]["content"])
print("\nFinish reason:", result["choices"][0]["finish_reason"])

Status: 200

Answer:
وفقاً للمصدر 1 والمصدر 2، يجب أن يكون النص على فترة التجربة صراحة في عقد العمل، ولا يجوز أن تزيد مدتها على 180 يوماً. (المادة 8 من المصدر 1، المادة 19 من المصدر 2). ولا تدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية. (المادة 8 من المصدر 1، المادة 19 من المصدر 2).

Finish reason: stop


In [41]:
user_prompt = f"""السؤال:
{query}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه."""

In [42]:
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 300,
        "chat_template_kwargs": {"enable_thinking": False}
    }
)

result = response.json()

print("Status:", response.status_code)
print("\nAnswer:")
print(result["choices"][0]["message"]["content"])
print("\nFinish reason:", result["choices"][0]["finish_reason"])

Status: 200

Answer:
لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى (المادة 8 من المصدر 1، والمادة 19 من المصدر 2).

Finish reason: stop


In [43]:
print(user_prompt)

السؤال:
كم مدة إجازة عيد الفطر للعمال؟

السياق القانوني المسترجع:
المصدر 1:
نوع المصدر: workplace_regulation
المادة: 8
الصفحة: 30

النص:
إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع
الأحوال على )مائة وثمانين( يوماً ، ولا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس،
والإجازة المرضية ، ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة .

النقل


المصدر 2:
نوع المصدر: executive_regulation
المادة: 19
الصفحة: 15

النص:
في تنفيذ حكم )المادة الثالثة والخمسون( من النظام:
1.يجب النص صـراحة في عقد العمل على تحديد مدة التجربة المتفق عليها على ألا تتجاوز مدة التجربة 180
يوماً ويجوز تقسـيم مدة التجربة إلى
مُدَد متعددة، شريطة أن تكون هذه المُدد متصلة، وأن يتم النص على ذلك بوضوح عند إبرام عقد العمل.
2. لا يدخل في حساب مدة التجربة إجازة عيدي الفطر والأضحى، وإجازة اليوم الوطني، وإجازة يوم التأسيس، والإجازة المرضية.


أجب عن السؤال بناءً على السياق أعلاه.


In [44]:
query = "ما هي حقوق المرأة العاملة في فترة الرضاعة؟"

print("Question:", query)

Question: ما هي حقوق المرأة العاملة في فترة الرضاعة؟


In [45]:
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    k=2
)

print("Retrieved sources:\n")

for rank, idx in enumerate(indices[0], start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"--- Rank {rank} ---")
    print("Score:", round(float(scores[0][rank - 1]), 4))
    print("Type:", metadata["article_type"])
    print("Article:", metadata["article"])
    print("Page:", metadata["page_start"])
    print()
    print(chunk["text"])
    print()

Retrieved sources:

--- Rank 1 ---
Score: 0.7577
Type: executive_regulation
Article: 31
Page: 22

في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـ

In [46]:
context_parts = []

for rank, idx in enumerate(indices[0][:2], start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    context_parts.append(
        f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
    )

context = "\n\n".join(context_parts)

print(context)

المصدر 1:
نوع المصدر: executive_regulation
المادة: 31
الصفحة: 22

النص:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بال

In [47]:
context_parts = []

for rank, idx in enumerate(indices[0], start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    context_parts.append(
        f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
    )

context = "\n\n".join(context_parts)

print(context)

المصدر 1:
نوع المصدر: executive_regulation
المادة: 31
الصفحة: 22

النص:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بال

In [48]:
system_prompt = """أنت مساعد متخصص في نظام العمل السعودي واللوائح التنفيذية المرتبطة به.

قواعد الإجابة الصارمة:

1. اعتمد فقط على النصوص القانونية الموجودة في السياق المسترجع.
2. لا تستخدم أي معلومة من معرفتك العامة خارج السياق.
3. أجب مباشرة عن سؤال المستخدم وباختصار.
4. لا تخلط بين الأحكام الواردة في مصادر مختلفة بطريقة تغيّر معناها.
5. إذا كانت هناك عدة مواد مرتبطة بالسؤال، اشرح حكم كل مادة بشكل منفصل إذا كان ذلك ضروريًا لفهم الإجابة.
6. لا تستنتج وجود حق أو التزام أو مدة أو استثناء إلا إذا كان النص المسترجع يدعمه بوضوح.
7. لا تعتبر حكمين متشابهين حكمًا واحدًا، ولا تضفهما إلى بعضهما إلا إذا نص السياق بوضوح على ذلك.
8. حافظ على أرقام وأسماء المواد كما وردت في السياق.
9. عند ذكر حكم قانوني، اذكر المادة ومصدرها.
10. إذا كان السياق المسترجع لا يحتوي على معلومات كافية للإجابة، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
11. أجب باللغة العربية فقط.
12. لا تستخدم لغة أخرى.
13. لا تكرر النص القانوني حرفيًا؛ لخّص الحكم المطلوب بدقة.
"""

In [49]:
user_prompt = f"""السؤال:
{query}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه.
أجب باللغة العربية فقط وباختصار."""

In [50]:
response = requests.post(
    "http://127.0.0.1:8080/v1/chat/completions",
    json={
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": 0.0,
        "max_tokens": 300
    }
)

print("Status:", response.status_code)

result = response.json()["choices"][0]["message"]["content"]

print("\nGemma answer:")
print(result)

Status: 200

Gemma answer:
يحق للمرأة العاملة الحصول على فترة أو فترات استراحة بقصد إرضاع مولودها، مدفوعة الأجر لا تزيد عن ساعة في اليوم الواحد لمدة أربعة وعشرين شهرًا من تاريخ الوضع (المادة 31 من اللائحة التنفيذية للنظام، والمادة الرابعة والخمسون بعد المائة من نظام العمل).


In [51]:
def ask_question(question, k=2):
    # 1. Embed the question
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # 2. Retrieve relevant chunks
    scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"),
        k=k
    )

    # 3. Build legal context + save source information
    context_parts = []
    sources = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = rag_chunks[idx]
        metadata = chunk["metadata"]

        context_parts.append(
            f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
        )

        sources.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "article_type": metadata["article_type"],
            "article": metadata["article"],
            "page": metadata["page_start"]
        })

    context = "\n\n".join(context_parts)

    # 4. Build the prompt
    user_prompt = f"""السؤال:
{question}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه.
أجب باللغة العربية فقط وباختصار."""

    # 5. Ask Gemma
    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.0,
            "max_tokens": 300
        }
    )

    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"]

    # 6. Return answer + sources
    return {
        "answer": answer,
        "sources": sources
    }

In [52]:
answer = ask_question("ما هي مدة فترة التجربة؟")
print(answer)

{'answer': 'لا تتجاوز مدة التجربة 180 يومًا (المادة 19 من اللائحة التنفيذية للنظام، والمادة 8 من لائحة بيئة العمل).', 'sources': [{'rank': 1, 'score': 0.6757864356040955, 'article_type': 'executive_regulation', 'article': '19', 'page': 15}, {'rank': 2, 'score': 0.6284093856811523, 'article_type': 'workplace_regulation', 'article': '8', 'page': 30}]}


In [53]:
answer = ask_question("ما هي حقوق المرأة العاملة في فترة الرضاعة؟")
print(answer)

{'answer': 'يحق للمرأة العاملة الحصول على فترة أو فترات استراحة بقصد إرضاع مولودها، مدفوعة الأجر لا تزيد عن ساعة في اليوم الواحد لمدة أربعة وعشرين شهرًا من تاريخ الوضع (المادة 31 من اللائحة التنفيذية للنظام، والمادة الرابعة والخمسون بعد المائة من نظام العمل).', 'sources': [{'rank': 1, 'score': 0.7577025294303894, 'article_type': 'executive_regulation', 'article': '31', 'page': 22}, {'rank': 2, 'score': 0.7028453946113586, 'article_type': 'labor_law', 'article': 'الرابعة والخمسون بعد المائة', 'page': 22}]}


In [54]:
answer = ask_question("ما هو الحد الأدنى لرأس مال الشركة؟")
print(answer)

{'answer': 'لا يحتوي السياق المسترجع على معلومات كافية للإجابة.', 'sources': [{'rank': 1, 'score': 0.4694746136665344, 'article_type': 'workplace_regulation', 'article': '21', 'page': 31}, {'rank': 2, 'score': 0.4098255932331085, 'article_type': 'workplace_regulation', 'article': '37', 'page': 34}]}


In [55]:
answer = ask_question("كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟")
print(answer)

{'answer': 'لا يجوز أن تزيد ساعات العمل الإضافية خلال السنة على سبعمائة وعشرين ساعة (المادة 22 من اللائحة التنفيذية).', 'sources': [{'rank': 1, 'score': 0.732741117477417, 'article_type': 'executive_regulation', 'article': '22', 'page': 17}, {'rank': 2, 'score': 0.6744836568832397, 'article_type': 'labor_law', 'article': 'السابعة بعد المائة', 'page': 17}]}


In [56]:
result = ask_question("كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟")

print("الإجابة:")
print(result["answer"])

print("\nالمصادر:")
for source in result["sources"]:
    print(
        f'- Rank {source["rank"]} | '
        f'Score: {source["score"]:.4f} | '
        f'{source["article_type"]} | '
        f'المادة: {source["article"]} | '
        f'الصفحة: {source["page"]}'
    )

الإجابة:
لا يجوز أن تزيد ساعات العمل الإضافية خلال السنة على سبعمائة وعشرين ساعة (المادة 22 من اللائحة التنفيذية).

المصادر:
- Rank 1 | Score: 0.7327 | executive_regulation | المادة: 22 | الصفحة: 17
- Rank 2 | Score: 0.6745 | labor_law | المادة: السابعة بعد المائة | الصفحة: 17


In [57]:
result = ask_question("هل يحق لصاحب العمل إنهاء عقد العامل خلال فترة التجربة؟")

print("الإجابة:")
print(result["answer"])

print("\nالمصادر:")
for source in result["sources"]:
    print(
        f'- Rank {source["rank"]} | '
        f'Score: {source["score"]:.4f} | '
        f'{source["article_type"]} | '
        f'المادة: {source["article"]} | '
        f'الصفحة: {source["page"]}'
    )

الإجابة:
نعم، يحق لصاحب العمل إنهاء عقد العامل خلال فترة التجربة، وذلك بموجب المادة الثالثة والخمسين من نظام العمل.

المصادر:
- Rank 1 | Score: 0.7761 | labor_law | المادة: الثالثة والخمسون | الصفحة: 15
- Rank 2 | Score: 0.7735 | workplace_regulation | المادة: 57 | الصفحة: 37


In [58]:
result = ask_question("ما هي مدة إجازة الوضع للمرأة العاملة وما هي حقوقها أثناء هذه الإجازة؟")

print("الإجابة:")
print(result["answer"])

print("\nالمصادر:")
for source in result["sources"]:
    print(
        f'- Rank {source["rank"]} | '
        f'Score: {source["score"]:.4f} | '
        f'{source["article_type"]} | '
        f'المادة: {source["article"]} | '
        f'الصفحة: {source["page"]}'
    )

الإجابة:
مدة إجازة الوضع للمرأة العاملة هي اثني عشر أسبوعًا بأجر كامل (المصدر 1، المادة 50، الصفحة 36). منها ستة أسابيع وجوبية بعد الوضع، ويمكن توزيع الأسابيع الستة المتبقية. يحق لها تمديد الإجازة شهرًا دون أجر (المصدر 1، المادة 50، الصفحة 36). عند العودة للعمل، يحق لها فترة أو فترات استراحة للإرضاع لا تزيد على الساعة في اليوم الواحد دون تخفيض الأجر (المصدر 2، المادة الرابعة والخمسون بعد المائة، الصفحة 22).

المصادر:
- Rank 1 | Score: 0.7968 | workplace_regulation | المادة: 50 | الصفحة: 36
- Rank 2 | Score: 0.7961 | labor_law | المادة: الرابعة والخمسون بعد المائة | الصفحة: 22


In [59]:
result = ask_question("كم مدة إجازة الوضع للمرأة العاملة؟")

print("الإجابة:")
print(result["answer"])

print("\nالمصادر:")
for source in result["sources"]:
    print(
        f'- Rank {source["rank"]} | '
        f'Score: {source["score"]:.4f} | '
        f'{source["article_type"]} | '
        f'المادة: {source["article"]} | '
        f'الصفحة: {source["page"]}'
    )

الإجابة:
مدة إجازة الوضع للمرأة العاملة هي اثني عشر أسبوعًا بأجر كامل (المصدر 1، المادة 50، الصفحة 36).

المصادر:
- Rank 1 | Score: 0.8378 | workplace_regulation | المادة: 50 | الصفحة: 36
- Rank 2 | Score: 0.7978 | labor_law | المادة: الرابعة والخمسون بعد المائة | الصفحة: 22


In [60]:
for i, chunk in enumerate(rag_chunks):
    text = chunk["text"]

    if any(word in text for word in ["وضع", "الأمومة", "الأم", "ولادة", "الولادة"]):
        print("Index:", i)
        print("Article:", chunk["metadata"]["article"])
        print("Type:", chunk["metadata"]["article_type"])
        print("Page:", chunk["metadata"]["page_start"])
        print()
        print(text)
        print("-" * 80)

Index: 6
Article: الثانية والعشرون
Type: labor_law
Page: 5

توفر الوزارة قنوات للتوظيف دون مقابل، تقوم بما يأتي:
1. مساعدة العمال في الحصول على الأعمال المناسبة، ومساعدة أصحاب الأعمال في إيجاد العمال المناسبين.
2. جمع المعلومات الضـرورية عن سـوق العمل وتطوره وتحليلها؛ لكي تكون في متناول مختلف الهيئات العامة والخاصـة المعنية بشـؤون
التخطيط الاقتصادي والاجتماعي.
3. تنفيذ الواجبات الآتية:
3/1- تسجيل طالبي العمل.
3/2- الحصول على بيانات بالأعمال الشاغرة من أصحاب الأعمال.
3/3- مواءمة طلبات طالبي العمل مع الأعمال الشاغرة وفق مؤهلاتهم.
3/
- تقـديم النصــح والمعونـة إلى طـالبي العمـل فيمـا يختص بـالتـأهيـل والتـدريـب المهني، أو بـإعـادة التـدريـب اللازم للحصــول على الأعمـال
الشاغرة.
3/5- غير ذلك من الأمور التي تقررها الوزارة.
--------------------------------------------------------------------------------
Index: 30
Article: الثامنة بعد المائة
Type: labor_law
Page: 17

لا تسري أحكام المادتين الثامنة والتسعين والأولى بعد المائة من هذا النظام على الحالات الآتية:
1.
الأشـخاص الذين يشـغلون مناصـب ع

In [61]:
with open(PROCESSED_DIR / "labor_law_articles.json", "r", encoding="utf-8") as f:
    labor_articles = json.load(f)

for article in labor_articles:
    text = article["text"]

    if any(word in text for word in ["وضع", "الأمومة", "ولادة", "الولادة"]):
        print("Article:", article["article"])
        print("Page:", article["page_start"])
        print()
        print(text)
        print("-" * 80)

In [62]:
!pip install PyMuPDF


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
import fitz

pdf_path = PROJECT_ROOT / "data" / "raw" / "اللائحة التنفيذية لنظام العمل وملحقاتها.pdf"

doc = fitz.open(pdf_path)

for page_num in range(len(doc)):
    text = doc[page_num].get_text()

    if "إجازة الوضع" in text or "الوضع" in text:
        print("=" * 80)
        print("PAGE:", page_num + 1)
        print(text[:5000])

PAGE: 36
 
35 
الرعاية الطبية 
المادة )49
( 
تقوم المنشـــأة بـالتـأمين على جميع العـاملين لـديهـا صــحيـاً ؛ وفقـاً لمـا يقرره نظـام التـأمين الصــحي التعـاوني ، ولائحتـه التنفيـذيـة ، كمـا تقوم 
بالاشتراك عن جميع العاملين في فرع الأخطار المهنية لدى المؤسسة العامة للتأمينات الاجتماعية ؛ وفقاً لما يقرره نظامها . 
 
 
 
أحكام خاصة بالمرأة 
المادة )50
( 
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع 
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة 
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها 
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر. 
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر( 
بأجر كامل ت

In [64]:
import fitz

pdf_path = PROJECT_ROOT / "data" / "raw" / "اللائحة التنفيذية لنظام العمل وملحقاتها.pdf"

doc = fitz.open(pdf_path)

for i, page in enumerate(doc):
    text = page.get_text()

    if "أحكام خاصة بالمرأة" in text:
        print("PDF page:", i + 1)
        print(text[:1000])

PDF page: 36
 
35 
الرعاية الطبية 
المادة )49
( 
تقوم المنشـــأة بـالتـأمين على جميع العـاملين لـديهـا صــحيـاً ؛ وفقـاً لمـا يقرره نظـام التـأمين الصــحي التعـاوني ، ولائحتـه التنفيـذيـة ، كمـا تقوم 
بالاشتراك عن جميع العاملين في فرع الأخطار المهنية لدى المؤسسة العامة للتأمينات الاجتماعية ؛ وفقاً لما يقرره نظامها . 
 
 
 
أحكام خاصة بالمرأة 
المادة )50
( 
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع 
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة 
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها 
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر. 
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر( 
بأجر كا

In [65]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI


In [66]:
import pymupdf

pdf_path = PROJECT_ROOT / "data" / "raw" / "اللائحة التنفيذية لنظام العمل وملحقاتها.pdf"

doc = pymupdf.open(pdf_path)

pages = [page.get_text() for page in doc]

print("Number of pages:", len(pages))
print("Page 36 contains Article 50:", "المادة )50" in pages[35] or "المادة (50" in pages[35])

Number of pages: 109
Page 36 contains Article 50: True


In [67]:
for page_num in range(26, 45):
    text = pages[page_num]
    
    print("=" * 80)
    print(f"PDF PAGE {page_num + 1}")
    print(text[:300].replace("\n", " | "))

PDF PAGE 27
  | 26  |   |   |   |   |    | ملحق رقم )1(  | النموذج الموحد للائحة تنظيم العمل  |   | 
PDF PAGE 28
  | 27  |   |   | لائحة تنظيم العمل  |   |   | مقدمة:  | وضــعـت هـذه اللائحـة تنفيـذاً لحكم الفقرة )1( من المـادة )الثـانيـة عشــرة( من نظـام العمـل الصـــادر بـالمرســوم الملكي رقم )م/51 | ( و تـاريخ  | 23 | /8/ | 1426 |  هـ وتعديلاته  .   |   | بيانات المنشأة  | اســـــم المنشـــــأة : ..........................................
PDF PAGE 29
  | 28  |   | أحكام عامة  | المادة )1(  | يقصد بالألفاظ والعبارات الآتية - أينما وردت في هذه اللائحة - المعاني الموضحة أمام كل منها مالم يقتض السياق خلاف ذلك :  | الوزارة : وزارة الموارد البشرية والتنمية الاجتماعية .  | النظام : نظام العمل الصادر بالمرسوم الملكي رقم )م/51 | ( و تاريخ 23 | /8/ | 1426 |  هـ وتعديلاته .   | ا
PDF PAGE 30
  | 29  | المادة )7(  | مع مراعاة التاريخ المحدد في عقد العمل لمباشرة العمل ؛ يحق للمنشأة إلغاء عقد العامل الذي لا يباشر مهام عمله دون عذر مشروع خلال  | سبعة أيام عمل من تاريخ التوقيع على العقد بين الطرفين

In [68]:
from pathlib import Path
import json

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Processed directory:", PROCESSED_DIR)

Processed directory: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed


In [69]:
import json

with open(PROCESSED_DIR / "labor_law_articles.json", "r", encoding="utf-8") as f:
    labor_articles = json.load(f)

with open(PROCESSED_DIR / "executive_regulations.json", "r", encoding="utf-8") as f:
    executive_articles = json.load(f)

print("Labor Law articles:", len(labor_articles))
print("Executive Regulation articles:", len(executive_articles))

print("\nLabor Law types:", sorted(set(a["article_type"] for a in labor_articles)))
print("Executive types:", sorted(set(a["article_type"] for a in executive_articles)))

Labor Law articles: 45
Executive Regulation articles: 45

Labor Law types: ['labor_law']
Executive types: ['executive_regulation']


In [70]:
print(json.dumps(labor_articles[0], ensure_ascii=False, indent=2))

{
  "section": "main_regulations",
  "article": "السادسة",
  "text": "تسري على العامل العرضي والموسمي والمؤقت الأحكام الخاصة بالواجبات وقواعد التأديب، والحد الأقصى لساعات العمل، وفترات\nالراحة اليومية والراحة الأسبوعية، والتشغيل الإضافي، والعطلات الرسمية، وقواعد السلامة، والصحة المهنية، وإصابات العمل\nوالتعويض عنها، وما يقرره الوزير.",
  "page_start": 3,
  "page_end": 3,
  "article_type": "labor_law",
  "related_regulations": [
    "1"
  ]
}


In [71]:
print(json.dumps(executive_articles[0], ensure_ascii=False, indent=2))

{
  "section": "main_regulations",
  "article": "1",
  "text": "في تنفيذ أحكام )المادة السادسة( من النظام:\n1.إذا اســتمر الطرفـان في كـل من عقـد العمـل المؤقـت أو عقـد العمـل العرضــي في العمـل لمـدة تجـاوزت تســعين يومـاً، تحول عقـد العمـل\nالمؤقت أو عقد العمل العرضـي إلى عقد عمل يخضـع لجميع أحكام نظام العمل دون النظر لمسـمى تأشـيرة العمل التي قدم بها العامل - إذا\nكان وافداً - ومدة الإقامة الممنوحة له بموجبها.\n2.تختص المحاكم العمالية بالنظر في الخلافات التي قد تنشأ بين الطرفين في كل من عقد العمل العرضي، والموسمي، والمؤقت.",
  "page_start": 3,
  "page_end": 3,
  "article_type": "executive_regulation",
  "related_law_articles": [
    "السادسة"
  ]
}


In [72]:
text = pages[35]

start = text.find("المادة )49")
print(text[start:start + 5000])

المادة )49
( 
تقوم المنشـــأة بـالتـأمين على جميع العـاملين لـديهـا صــحيـاً ؛ وفقـاً لمـا يقرره نظـام التـأمين الصــحي التعـاوني ، ولائحتـه التنفيـذيـة ، كمـا تقوم 
بالاشتراك عن جميع العاملين في فرع الأخطار المهنية لدى المؤسسة العامة للتأمينات الاجتماعية ؛ وفقاً لما يقرره نظامها . 
 
 
 
أحكام خاصة بالمرأة 
المادة )50
( 
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع 
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة 
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها 
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر. 
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر( 
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع،

In [73]:
for page_num in range(27, 39):
    text = pages[page_num]

    matches = []
    for line in text.splitlines():
        line = line.strip()

        if "المادة" in line:
            matches.append(line)

    print(f"\nPDF PAGE {page_num + 1}")
    for match in matches:
        print(match)


PDF PAGE 28

PDF PAGE 29
المادة )1(
المادة )2(
المادة )3(
المادة )4(
المادة )5(
المادة )6(

PDF PAGE 30
المادة )7(
المادة )8(
المادة )9(
المادة )10
المادة )11(
عند انتهاء خدمة العامل ، طبقاً لأحكام المادة )الأربعون( فقرة ) 1 ( من النظام .
المادة )12
المادة )13

PDF PAGE 31
المادة )14
إذا استقال العامل من العمل ، أو تركه لغير الحالات الواردة في المادة )الحادية والثمانون( من النظام أثناء فترة التدريب أو التأهيل .
المادة )15
المادة )16
المادة )17
المادة )18
المادة )19
المادة )20
المادة )21

PDF PAGE 32
المادة )22
المادة )23
المادة )24
المادة )26
المادة )27
المادة )28

PDF PAGE 33
المادة )29
يبين فيه عدد السـاعات الإضـافية المكلف بها العامل ، وعدد الأيام اللازمة لذلك ؛ وفق ما نصـَّت عليه المادة )السـادسـة بعد المائة( من
المادة )30
وتوضح اللائحة التنفيذية للنظام الأعمال المبينة في الفقرات 2، 3، 4 من هذه المادة والحد الأقصى لساعات العمل فيها .
المادة )31
المادة )32
المادة )33
المادة )34
المادة )35
المادة )36

PDF PAGE 34
المادة )37
المادة )38
المادة )39
المادة )40
المادة )41
المادة )42

PDF

In [74]:
import re
import json

# The model workplace regulation starts on PDF page 28
# and ends before Appendix 2 on PDF page 44.
workplace_pages = pages[27:43]

article_pattern = re.compile(
    r"المادة\s*\)?\s*\(\s*(\d+)\s*\)\s*\)?"
)

workplace_articles = []

current_article = None
current_text = []
current_start_page = None

for page_idx, page_text in enumerate(workplace_pages, start=28):
    lines = page_text.splitlines()

    for line in lines:
        line = line.strip()

        match = article_pattern.search(line)

        if match:
            # Save the previous article
            if current_article is not None:
                workplace_articles.append({
                    "section": "workplace_regulation",
                    "article": current_article,
                    "text": "\n".join(current_text).strip(),
                    "page_start": current_start_page,
                    "page_end": page_idx,
                    "article_type": "workplace_regulation"
                })

            # Start new article
            current_article = match.group(1)
            current_text = []

            # Keep any text appearing after the article heading
            remainder = line[match.end():].strip()
            if remainder:
                current_text.append(remainder)

            current_start_page = page_idx

        elif current_article is not None:
            current_text.append(line)

# Save final article
if current_article is not None:
    workplace_articles.append({
        "section": "workplace_regulation",
        "article": current_article,
        "text": "\n".join(current_text).strip(),
        "page_start": current_start_page,
        "page_end": 43,
        "article_type": "workplace_regulation"
    })

print("Extracted workplace regulation articles:", len(workplace_articles))

print("\nArticle numbers:")
print([a["article"] for a in workplace_articles])

Extracted workplace regulation articles: 0

Article numbers:
[]


In [75]:
import re

workplace_pages = pages[27:43]

# Match OCR format like:
# المادة )50
# (
article_pattern = re.compile(r"المادة\s*\)?\s*(\d+)\s*")

workplace_articles = []

current_article = None
current_text = []
current_start_page = None

for page_idx, page_text in enumerate(workplace_pages, start=28):
    for line in page_text.splitlines():
        line = line.strip()

        match = article_pattern.search(line)

        if match:
            if current_article is not None:
                workplace_articles.append({
                    "section": "workplace_regulation",
                    "article": current_article,
                    "text": "\n".join(current_text).strip(),
                    "page_start": current_start_page,
                    "page_end": page_idx,
                    "article_type": "workplace_regulation"
                })

            current_article = match.group(1)
            current_text = []
            current_start_page = page_idx

            remainder = line[match.end():].strip()
            if remainder:
                current_text.append(remainder)

        elif current_article is not None:
            current_text.append(line)

# Save final article
if current_article is not None:
    workplace_articles.append({
        "section": "workplace_regulation",
        "article": current_article,
        "text": "\n".join(current_text).strip(),
        "page_start": current_start_page,
        "page_end": 43,
        "article_type": "workplace_regulation"
    })

print("Extracted:", len(workplace_articles))
print("Articles:", [a["article"] for a in workplace_articles])

Extracted: 71
Articles: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72']


In [76]:
article_50 = next(
    a for a in workplace_articles
    if a["article"] == "50"
)

print(json.dumps(article_50, ensure_ascii=False, indent=2))

{
  "section": "workplace_regulation",
  "article": "50",
  "text": "(\n1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع\nالسـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة\nمن جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها\nإجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.\n2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(\nبأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة لمدة شهر دون أجر .",
  "page_start": 36,
  "page_end": 36,
  "article_type": "workplace_regulation"
}


In [77]:
# Clean OCR artifacts from the beginning of article text
for article in workplace_articles:
    article["text"] = article["text"].lstrip()

    if article["text"].startswith("("):
        article["text"] = article["text"][1:].lstrip()

# Save the new dataset
output_path = PROCESSED_DIR / "workplace_regulation_articles.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(workplace_articles, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)
print("Articles saved:", len(workplace_articles))

# Verify Article 50
article_50 = next(a for a in workplace_articles if a["article"] == "50")

print("\nArticle 50:")
print(article_50["text"])

Saved: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\workplace_regulation_articles.json
Articles saved: 71

Article 50:
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة لمدة شهر دون أجر .


In [78]:
# Build RAG chunks from all three source types

import json

# Combine all legal sources
all_articles = (
    labor_articles
    + executive_articles
    + workplace_articles
)

print("Total articles:", len(all_articles))
print("Labor Law:", len(labor_articles))
print("Executive Regulations:", len(executive_articles))
print("Workplace Regulation:", len(workplace_articles))


# Create chunks
rag_chunks = []

for article in all_articles:
    text = article["text"].strip()

    # Keep each article as one chunk for now
    chunk = {
        "text": text,
        "metadata": {
            "section": article["section"],
            "article": article["article"],
            "article_type": article["article_type"],
            "page_start": article["page_start"],
            "page_end": article["page_end"]
        }
    }

    # Preserve relationship fields when available
    if "related_regulations" in article:
        chunk["metadata"]["related_regulations"] = article["related_regulations"]

    if "related_law_articles" in article:
        chunk["metadata"]["related_law_articles"] = article["related_law_articles"]

    rag_chunks.append(chunk)


print("Total RAG chunks:", len(rag_chunks))

# Save
output_path = PROCESSED_DIR / "rag_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)

Total articles: 161
Labor Law: 45
Executive Regulations: 45
Workplace Regulation: 71
Total RAG chunks: 161
Saved: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\rag_chunks.json


In [79]:
# Verify the new RAG dataset

workplace_chunks = [
    chunk for chunk in rag_chunks
    if chunk["metadata"]["article_type"] == "workplace_regulation"
]

print("Total chunks:", len(rag_chunks))
print("Workplace regulation chunks:", len(workplace_chunks))

article_50_chunk = next(
    chunk for chunk in rag_chunks
    if chunk["metadata"]["article_type"] == "workplace_regulation"
    and chunk["metadata"]["article"] == "50"
)

print("\nArticle 50:")
print(article_50_chunk["text"])

Total chunks: 161
Workplace regulation chunks: 71

Article 50:
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة لمدة شهر دون أجر .


In [80]:
# Regenerate embeddings for the updated RAG dataset

import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device="cuda"
)

texts = [chunk["text"] for chunk in rag_chunks]

rag_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", rag_embeddings.shape)
print("Device:", embedding_model.device)

# Save embeddings
embeddings_path = PROCESSED_DIR / "rag_embeddings.npy"
np.save(embeddings_path, rag_embeddings)

print("Saved:", embeddings_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding shape: (161, 768)
Device: cuda:0
Saved: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\rag_embeddings.npy


In [81]:
# Rebuild FAISS index for the updated embeddings

import faiss
import numpy as np

# Make sure embeddings are float32
embeddings = np.asarray(rag_embeddings, dtype="float32")

# Create inner-product index
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])

# Add all embeddings
faiss_index.add(embeddings)

print("FAISS vectors:", faiss_index.ntotal)
print("Embedding dimension:", faiss_index.d)

# Save the updated index
faiss_path = PROCESSED_DIR / "faiss.index"
faiss.write_index(faiss_index, str(faiss_path))

print("Saved:", faiss_path)

FAISS vectors: 161
Embedding dimension: 768
Saved: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\faiss.index


In [82]:
# Test retrieval for maternity leave

query = "كم مدة إجازة الوضع للمرأة العاملة؟"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    k=5
)

print("Question:", query)
print("\nTop 5 retrieved sources:\n")

for rank, idx in enumerate(indices[0], start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(f"--- Rank {rank} ---")
    print("Score:", round(float(scores[0][rank - 1]), 4))
    print("Type:", metadata["article_type"])
    print("Article:", metadata["article"])
    print("Page:", metadata["page_start"])
    print("Text:")
    print(chunk["text"][:700])
    print()

Question: كم مدة إجازة الوضع للمرأة العاملة؟

Top 5 retrieved sources:

--- Rank 1 ---
Score: 0.8378
Type: workplace_regulation
Article: 50
Page: 36
Text:
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة ل

--- Rank 2 ---
Score: 0.7978
Type: labor_law
Article: الرابعة والخمسون بعد المائة
Page: 22
Text:
يحق للمرأة العاملة عندما تعود إلى مزاولة عملها

In [83]:
system_prompt = """أنت مساعد متخصص في نظام العمل السعودي واللوائح التنفيذية المرتبطة به.

قواعد الإجابة الصارمة:

1. اعتمد فقط على النصوص القانونية الموجودة في السياق المسترجع.
2. لا تستخدم أي معلومة من معرفتك العامة خارج السياق.
3. أجب مباشرة عن سؤال المستخدم وباختصار.
4. لا تخلط بين الأحكام الواردة في مصادر مختلفة بطريقة تغيّر معناها.
5. إذا كانت هناك عدة مواد مرتبطة بالسؤال، اشرح حكم كل مادة بشكل منفصل إذا كان ذلك ضروريًا لفهم الإجابة.
6. لا تستنتج وجود حق أو التزام أو مدة أو استثناء إلا إذا كان النص المسترجع يدعمه بوضوح.
7. لا تعتبر حكمين متشابهين حكمًا واحدًا، ولا تضفهما إلى بعضهما إلا إذا نص السياق بوضوح على ذلك.
8. حافظ على أرقام وأسماء المواد كما وردت في السياق.
9. عند ذكر حكم قانوني، اذكر المادة ومصدرها.
10. إذا كان السياق المسترجع لا يحتوي على معلومات كافية للإجابة، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
11. أجب باللغة العربية فقط.
12. لا تستخدم لغة أخرى.
13. لا تكرر النص القانوني حرفيًا؛ لخّص الحكم المطلوب بدقة.
"""

print("System prompt loaded.")

System prompt loaded.


In [84]:
def ask_question(question, k=2):
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"),
        k=k
    )

    context_parts = []
    sources = []

    for rank, idx in enumerate(indices[0], start=1):
        chunk = rag_chunks[idx]
        metadata = chunk["metadata"]

        context_parts.append(
            f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
        )

        sources.append({
            "rank": rank,
            "score": float(scores[0][rank - 1]),
            "article_type": metadata["article_type"],
            "article": metadata["article"],
            "page": metadata["page_start"]
        })

    context = "\n\n".join(context_parts)

    user_prompt = f"""السؤال:
{question}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه.
أجب باللغة العربية فقط وباختصار."""

    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.0,
            "max_tokens": 300
        }
    )

    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"]

    return {
        "answer": answer,
        "sources": sources
    }

In [85]:
import requests

In [86]:
result = ask_question("كم مدة إجازة الوضع للمرأة العاملة؟", k=2)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
مدة إجازة الوضع للمرأة العاملة هي اثني عشر أسبوعًا بأجر كامل (المصدر 1، المادة 50، الصفحة 36).

Sources:
{'rank': 1, 'score': 0.837786853313446, 'article_type': 'workplace_regulation', 'article': '50', 'page': 36}
{'rank': 2, 'score': 0.7978457808494568, 'article_type': 'labor_law', 'article': 'الرابعة والخمسون بعد المائة', 'page': 22}


In [87]:
# Inspect retrieval scores for maternity leave

query = "كم مدة إجازة الوضع للمرأة العاملة؟"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = faiss_index.search(
    np.array(query_embedding, dtype="float32"),
    k=10
)

for rank, idx in enumerate(indices[0], start=1):
    chunk = rag_chunks[idx]
    metadata = chunk["metadata"]

    print(
        f"{rank}. "
        f"score={scores[0][rank-1]:.4f} | "
        f"type={metadata['article_type']} | "
        f"article={metadata['article']} | "
        f"page={metadata['page_start']}"
    )

1. score=0.8378 | type=workplace_regulation | article=50 | page=36
2. score=0.7978 | type=labor_law | article=الرابعة والخمسون بعد المائة | page=22
3. score=0.7211 | type=executive_regulation | article=31 | page=22
4. score=0.6905 | type=labor_law | article=السادسة عشرة بعد المائة | page=19
5. score=0.6862 | type=workplace_regulation | article=33 | page=33
6. score=0.6721 | type=workplace_regulation | article=44 | page=35
7. score=0.6604 | type=workplace_regulation | article=34 | page=33
8. score=0.6550 | type=workplace_regulation | article=38 | page=34
9. score=0.6386 | type=workplace_regulation | article=8 | page=30
10. score=0.6331 | type=workplace_regulation | article=40 | page=34


In [88]:
# Test several maternity-related questions

test_questions = [
    "كم مدة إجازة الوضع للمرأة العاملة؟",
    "كم أسبوع إجازة الوضع؟",
    "ما هي حقوق المرأة أثناء إجازة الوضع؟",
    "متى تبدأ إجازة الوضع؟",
    "هل إجازة الوضع بأجر كامل؟"
]

for query in test_questions:
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"),
        k=3
    )

    print("\n" + "=" * 70)
    print("السؤال:", query)

    for rank, idx in enumerate(indices[0], start=1):
        metadata = rag_chunks[idx]["metadata"]

        print(
            f"{rank}. "
            f"score={scores[0][rank-1]:.4f} | "
            f"{metadata['article_type']} | "
            f"المادة {metadata['article']} | "
            f"صفحة {metadata['page_start']}"
        )


السؤال: كم مدة إجازة الوضع للمرأة العاملة؟
1. score=0.8378 | workplace_regulation | المادة 50 | صفحة 36
2. score=0.7978 | labor_law | المادة الرابعة والخمسون بعد المائة | صفحة 22
3. score=0.7211 | executive_regulation | المادة 31 | صفحة 22

السؤال: كم أسبوع إجازة الوضع؟
1. score=0.6471 | executive_regulation | المادة 26 | صفحة 19
2. score=0.6247 | workplace_regulation | المادة 17 | صفحة 31
3. score=0.6163 | workplace_regulation | المادة 50 | صفحة 36

السؤال: ما هي حقوق المرأة أثناء إجازة الوضع؟
1. score=0.7724 | executive_regulation | المادة 31 | صفحة 22
2. score=0.7449 | labor_law | المادة الرابعة والخمسون بعد المائة | صفحة 22
3. score=0.7384 | workplace_regulation | المادة 50 | صفحة 36

السؤال: متى تبدأ إجازة الوضع؟
1. score=0.5702 | executive_regulation | المادة 26 | صفحة 19
2. score=0.5633 | workplace_regulation | المادة 39 | صفحة 34
3. score=0.5343 | workplace_regulation | المادة 8 | صفحة 30

السؤال: هل إجازة الوضع بأجر كامل؟
1. score=0.6872 | workplace_regulation | المادة 38 | ص

In [89]:
# Check keyword matches for maternity-related terms

keywords = [
    "إجازة الوضع",
    "إجازة وضـع",
    "بأجر كامل",
    "الوضع",
    "المرأة العاملة"
]

for keyword in keywords:
    matches = []

    for chunk in rag_chunks:
        text = chunk["text"]
        if keyword in text:
            metadata = chunk["metadata"]

            matches.append(
                (
                    metadata["article_type"],
                    metadata["article"],
                    metadata["page_start"]
                )
            )

    print(f"\nKeyword: {keyword}")
    print("Matches:", len(matches))

    for match in matches[:10]:
        print(" -", match)


Keyword: إجازة الوضع
Matches: 1
 - ('workplace_regulation', '50', 36)

Keyword: إجازة وضـع
Matches: 1
 - ('workplace_regulation', '50', 36)

Keyword: بأجر كامل
Matches: 7
 - ('labor_law', 'الثانية عشرة بعد المائة', 18)
 - ('workplace_regulation', '33', 33)
 - ('workplace_regulation', '39', 34)
 - ('workplace_regulation', '40', 34)
 - ('workplace_regulation', '41', 34)
 - ('workplace_regulation', '43', 35)
 - ('workplace_regulation', '50', 36)

Keyword: الوضع
Matches: 1
 - ('workplace_regulation', '50', 36)

Keyword: المرأة العاملة
Matches: 2
 - ('executive_regulation', '31', 22)
 - ('workplace_regulation', '51', 36)


In [90]:
# Arabic text normalization for hybrid retrieval

import re

def normalize_arabic(text):
    text = text.lower()

    # Remove Arabic tatweel / kashida
    text = text.replace("ـ", "")

    # Normalize Alef variants
    text = re.sub(r"[إأآٱ]", "ا", text)

    # Normalize Hamza forms
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    # Normalize Yeh
    text = text.replace("ى", "ي")

    # Remove diacritics
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Quick test
tests = [
    "إجازة الوضع",
    "إجازة وضـع",
    "بأجر كامل"
]

for text in tests:
    print(f"Original:   {text}")
    print(f"Normalized: {normalize_arabic(text)}")
    print()

Original:   إجازة الوضع
Normalized: اجازة الوضع

Original:   إجازة وضـع
Normalized: اجازة وضع

Original:   بأجر كامل
Normalized: باجر كامل



In [91]:
# Add normalized text to every RAG chunk

for chunk in rag_chunks:
    chunk["normalized_text"] = normalize_arabic(chunk["text"])

print("Normalized text added to:", len(rag_chunks), "chunks")

# Verify Article 50
article_50 = next(
    chunk for chunk in rag_chunks
    if chunk["metadata"]["article_type"] == "workplace_regulation"
    and chunk["metadata"]["article"] == "50"
)

print("\nArticle 50 normalized text:")
print(article_50["normalized_text"][:500])

Normalized text added to: 161 chunks

Article 50 normalized text:
1- للمراة العاملة الحق في اجازة وضع باجر كامل لمدة) اثني عشر( اسبوعا، منها وجوبيا الاسابيع الستة التالية للوضع، ولها ان توزع الاسابيع الستة المتبقية وفق ما تراه، ابتداء من اربعة اسابيع قبل التاريخ المرجح للوضع. ويحدد التاريخ المرجح للوضع بموجب شهادة طبية مصدقة من جهة صحية. وفي حال قل المتبقي من مدة الاجازة عن )ستة( اسابيع نتيجة تاخر الوضع عن تاريخه المرجح، فتحتسب المدة المكملة لها اجازة دون اجر. وفي جميع الاحوال يحق للمراة العاملة تمديد هذه الاجازة )شهرا( دون اجر. 2- للمراة العاملة -في حالة انجا


In [92]:
# Hybrid retrieval: semantic similarity + exact keyword matching

def hybrid_search(question, k=5):
    # Normalize the question
    normalized_question = normalize_arabic(question)

    # Semantic retrieval
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    semantic_scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"),
        len(rag_chunks)
    )

    # Convert FAISS results into a dictionary
    semantic_score_map = {
        int(idx): float(score)
        for idx, score in zip(indices[0], semantic_scores[0])
    }

    # Extract meaningful query words
    query_words = set(
        word for word in normalized_question.split()
        if len(word) >= 3
    )

    results = []

    for idx, chunk in enumerate(rag_chunks):
        normalized_text = chunk["normalized_text"]

        # Word-level keyword matching
        matched_words = [
            word for word in query_words
            if word in normalized_text
        ]

        keyword_score = (
            len(matched_words) / len(query_words)
            if query_words
            else 0.0
        )

        semantic_score = semantic_score_map.get(idx, 0.0)

        # Hybrid score
        hybrid_score = (
            0.7 * semantic_score
            + 0.3 * keyword_score
        )

        results.append({
            "idx": idx,
            "semantic_score": semantic_score,
            "keyword_score": keyword_score,
            "hybrid_score": hybrid_score,
            "matched_words": matched_words
        })

    # Sort by hybrid score
    results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return results[:k]

In [93]:
query = "كم مدة إجازة الوضع للمرأة العاملة؟"

results = hybrid_search(query, k=5)

print("Question:", query)
print("\nHybrid retrieval:\n")

for rank, result in enumerate(results, start=1):
    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print(
        f"{rank}. "
        f"hybrid={result['hybrid_score']:.4f} | "
        f"semantic={result['semantic_score']:.4f} | "
        f"keyword={result['keyword_score']:.4f} | "
        f"type={metadata['article_type']} | "
        f"article={metadata['article']} | "
        f"page={metadata['page_start']} | "
        f"matched={result['matched_words']}"
    )

Question: كم مدة إجازة الوضع للمرأة العاملة؟

Hybrid retrieval:

1. hybrid=0.8265 | semantic=0.8378 | keyword=0.8000 | type=workplace_regulation | article=50 | page=36 | matched=['اجازة', 'الوضع', 'للمراة', 'مدة']
2. hybrid=0.7448 | semantic=0.7211 | keyword=0.8000 | type=executive_regulation | article=31 | page=22 | matched=['اجازة', 'الوضع', 'للمراة', 'مدة']
3. hybrid=0.7385 | semantic=0.7978 | keyword=0.6000 | type=labor_law | article=الرابعة والخمسون بعد المائة | page=22 | matched=['اجازة', 'الوضع', 'للمراة']
4. hybrid=0.6331 | semantic=0.5616 | keyword=0.8000 | type=workplace_regulation | article=51 | page=36 | matched=['اجازة', 'الوضع', 'للمراة', 'مدة']
5. hybrid=0.6033 | semantic=0.6905 | keyword=0.4000 | type=labor_law | article=السادسة عشرة بعد المائة | page=19 | matched=['اجازة', 'مدة']


In [94]:
def ask_question_hybrid(question, k=3):
    results = hybrid_search(question, k=k)

    context_parts = []
    sources = []

    for rank, result in enumerate(results, start=1):
        chunk = rag_chunks[result["idx"]]
        metadata = chunk["metadata"]

        context_parts.append(
            f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
        )

        sources.append({
            "rank": rank,
            "score": result["hybrid_score"],
            "article_type": metadata["article_type"],
            "article": metadata["article"],
            "page": metadata["page_start"]
        })

    context = "\n\n".join(context_parts)

    user_prompt = f"""السؤال:
{question}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه.
أجب باللغة العربية فقط وباختصار."""

    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.0,
            "max_tokens": 300
        }
    )

    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"]

    return {
        "answer": answer,
        "sources": sources
    }

In [95]:
result = ask_question_hybrid(
    "كم مدة إجازة الوضع للمرأة العاملة؟"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
مدة إجازة الوضع للمرأة العاملة هي اثني عشر أسبوعًا بأجر كامل (المصدر 1، المادة 50).

Sources:
{'rank': 1, 'score': 0.8264507973194122, 'article_type': 'workplace_regulation', 'article': '50', 'page': 36}
{'rank': 2, 'score': 0.7447723174095153, 'article_type': 'executive_regulation', 'article': '31', 'page': 22}
{'rank': 3, 'score': 0.7384920465946196, 'article_type': 'labor_law', 'article': 'الرابعة والخمسون بعد المائة', 'page': 22}


In [96]:
result = ask_question_hybrid(
    "هل إجازة الوضع للمرأة العاملة بأجر كامل؟ وكم مدتها؟"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
إجازة الوضع للمرأة العاملة بأجر كامل لمدة اثني عشر أسبوعًا (المصدر 1، المادة 50). يمكن تمديدها شهرًا دون أجر (المصدر 1، المادة 50).

Sources:
{'rank': 1, 'score': 0.783582067489624, 'article_type': 'workplace_regulation', 'article': '50', 'page': 36}
{'rank': 2, 'score': 0.7074767112731933, 'article_type': 'labor_law', 'article': 'الرابعة والخمسون بعد المائة', 'page': 22}
{'rank': 3, 'score': 0.6665821254253388, 'article_type': 'executive_regulation', 'article': '31', 'page': 22}


In [97]:
result = ask_question_hybrid(
    "ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
يحق للمرأة العاملة إجازة مدتها شهر بأجر كامل بعد انتهاء إجازة الوضع إذا أنجبت طفلاً مريضًا أو من ذوي الإعاقة ويتطلب الأمر مرافقًا مستمرًا، ويجوز تمديدها لمدة شهر دون أجر (المصدر 1، المادة 2).

Sources:
{'rank': 1, 'score': 0.5239895045757293, 'article_type': 'workplace_regulation', 'article': '50', 'page': 36}
{'rank': 2, 'score': 0.4895492374897003, 'article_type': 'executive_regulation', 'article': '31', 'page': 22}
{'rank': 3, 'score': 0.47113473415374757, 'article_type': 'executive_regulation', 'article': '9', 'page': 6}


In [98]:
query = "ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟"

results = hybrid_search(query, k=3)

for rank, result in enumerate(results, start=1):
    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print(f"\n{'='*60}")
    print(f"المصدر {rank}")
    print("النوع:", metadata["article_type"])
    print("المادة:", metadata["article"])
    print("الصفحة:", metadata["page_start"])
    print("النص:")
    print(chunk["text"])


المصدر 1
النوع: workplace_regulation
المادة: 50
الصفحة: 36
النص:
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة لمدة شهر دون أجر .

المصدر 2
النوع: executive_regulation
المادة: 31
الصفحة: 22
النص:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة 

In [99]:
system_prompt = """أنت مساعد متخصص في نظام العمل السعودي واللوائح التنفيذية المرتبطة به.

قواعد الإجابة الصارمة:

1. اعتمد فقط على النصوص القانونية الموجودة في السياق المسترجع.
2. لا تستخدم أي معلومة من معرفتك العامة خارج السياق.
3. أجب مباشرة عن سؤال المستخدم وباختصار.
4. لا تخلط بين الأحكام الواردة في مصادر مختلفة بطريقة تغيّر معناها.
5. إذا كانت هناك عدة مواد مرتبطة بالسؤال، اشرح حكم كل مادة بشكل منفصل إذا كان ذلك ضروريًا لفهم الإجابة.
6. لا تستنتج وجود حق أو التزام أو مدة أو استثناء إلا إذا كان النص المسترجع يدعمه بوضوح.
7. لا تعتبر حكمين متشابهين حكمًا واحدًا، ولا تضفهما إلى بعضهما إلا إذا نص السياق بوضوح على ذلك.
8. حافظ على أرقام وأسماء المواد كما وردت في السياق.
9. عند ذكر حكم قانوني، اذكر رقم المادة ومصدرها بدقة.
10. انتبه إلى أن الأرقام مثل "1-" و"2-" داخل نص المادة هي أرقام فقرات، وليست أرقام مواد جديدة.
11. إذا استندت إلى الفقرة الثانية من المادة 50 مثلًا، اذكرها بصيغة: "المادة 50، الفقرة 2"، وليس "المادة 2".
12. لا تنسب أي حكم إلى مادة أخرى لمجرد أن رقمًا ظهر داخل نص المادة.
13. إذا كان السياق المسترجع لا يحتوي على معلومات كافية للإجابة، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
14. أجب باللغة العربية فقط.
15. لا تستخدم لغة أخرى.
16. لا تكرر النص القانوني حرفيًا؛ لخّص الحكم المطلوب بدقة.
"""

In [100]:
result = ask_question_hybrid(
    "ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟"
)

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
يحق للمرأة العاملة إجازة مدتها شهر بأجر كامل بعد انتهاء إجازة الوضع إذا أنجبت طفلًا مريضًا أو من ذوي الإعاقة ويتطلب الأمر مرافقًا مستمرًا، ويجوز تمديدها لمدة شهر دون أجر (المادة 50، الفقرة 2).

Sources:
{'rank': 1, 'score': 0.5239895045757293, 'article_type': 'workplace_regulation', 'article': '50', 'page': 36}
{'rank': 2, 'score': 0.4895492374897003, 'article_type': 'executive_regulation', 'article': '31', 'page': 22}
{'rank': 3, 'score': 0.47113473415374757, 'article_type': 'executive_regulation', 'article': '9', 'page': 6}


In [101]:
print("Total RAG chunks:", len(rag_chunks))

print("\nArticle types:")
from collections import Counter

print(Counter(
    chunk["metadata"]["article_type"]
    for chunk in rag_chunks
))

Total RAG chunks: 161

Article types:
Counter({'workplace_regulation': 71, 'labor_law': 45, 'executive_regulation': 45})


In [102]:
evaluation_questions = [
    "كم مدة إجازة الوضع للمرأة العاملة؟",
    "هل إجازة الوضع للمرأة العاملة بأجر كامل؟",
    "ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟",
    "كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟",
    "ما الحد الأقصى لمدة فترة التجربة؟",
    "ما هي حقوق المرأة العاملة في فترة الرضاعة؟",
    "هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟",
    "ما المقصود بالشخص ذي الإعاقة في اللائحة؟",
    "ما هي مدة إجازة الوضع للمرأة العاملة إذا تأخر موعد الولادة عن التاريخ المرجح؟",
    "ما هو الحد الأدنى لرأس مال الشركة؟"
]

print("Evaluation questions:", len(evaluation_questions))

for i, question in enumerate(evaluation_questions, start=1):
    print(f"{i}. {question}")

Evaluation questions: 10
1. كم مدة إجازة الوضع للمرأة العاملة؟
2. هل إجازة الوضع للمرأة العاملة بأجر كامل؟
3. ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟
4. كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟
5. ما الحد الأقصى لمدة فترة التجربة؟
6. ما هي حقوق المرأة العاملة في فترة الرضاعة؟
7. هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟
8. ما المقصود بالشخص ذي الإعاقة في اللائحة؟
9. ما هي مدة إجازة الوضع للمرأة العاملة إذا تأخر موعد الولادة عن التاريخ المرجح؟
10. ما هو الحد الأدنى لرأس مال الشركة؟


In [103]:
for i, question in enumerate(evaluation_questions, start=1):
    results = hybrid_search(question, k=1)
    result = results[0]

    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print("\n" + "=" * 70)
    print(f"Question {i}: {question}")
    print(f"Type: {metadata['article_type']}")
    print(f"Article: {metadata['article']}")
    print(f"Page: {metadata['page_start']}")
    print(f"Score: {result['hybrid_score']:.4f}")
    print("\nText:")
    print(chunk["text"])


Question 1: كم مدة إجازة الوضع للمرأة العاملة؟
Type: workplace_regulation
Article: 50
Page: 36
Score: 0.8265

Text:
1- للمرأة العاملة الحق في إجازة وضـع بأجر كامل لمدة) اثني عشـر( أسـبوعاً، منها وجوبيًّا الأسـابيع السـتة التالية للوضـع، ولها أن توزع الأسـابيع
السـتة المتبقية وفق ما تراه، ابتداءً من أربعة أسـابيع قبل التاريخ المرجح للوضـع. ويحدد التاريخ المرجح للوضـع بموجب شـهادة طبية مصـدقة
من جهة صـحية. وفي حال قل المتبقي من مدة الإجازة عن )سـتة( أسـابيع نتيجة تأخر الوضـع عن تاريخه المرجح، فتحتسـب المدة المكملة لها
إجازة دون أجر. وفي جميع الأحوال يحق للمرأة العاملة تمديد هذه الإجازة )شهراً( دون أجر.
2- للمرأة العـاملـة -في حالة إنجـاب طفـل مريض أو من ذوي الإعاقة وتتطلـب حالته الصــحية مرافقاً مســتمراً له- الحق في إجازة مدتها )شــهر(
بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، ولها الحق في تمديد الإجازة لمدة شهر دون أجر .

Question 2: هل إجازة الوضع للمرأة العاملة بأجر كامل؟
Type: workplace_regulation
Article: 50
Page: 36
Score: 0.8248

Text:
1- للمرأة العاملة الحق في إجازة وضـع بأجر 

In [104]:
id="check_article_31"

In [105]:
article_31 = next(
    article
    for article in executive_articles
    if article["article"] == "31"
)

print("Article:", article_31["article"])
print("Page start:", article_31["page_start"])
print("Page end:", article_31["page_end"])
print("\nText:")
print(article_31["text"])

Article: 31
Page start: 22
Page end: 22

Text:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بالوقت الذي تحدده لفترة أو ف

In [106]:
for article in executive_articles:
    if "الحادية والستون" in article["text"] or article["article"] in ["61", "161"]:
        print("Article:", article["article"])
        print("Page:", article["page_start"], "-", article["page_end"])
        print("Text:")
        print(article["text"])
        print("\n" + "=" * 60)

Article: 21
Page: 16 - 16
Text:
في تنفيذ أحكام )المادة الحادية والستون( من النظام:
1.يقصـد بعبارة "تشـغيل العامل سـخرة": جميع الأعمال، أو الخدمات التي تفرض عنوة على العامل، ولم ينص عليها في عقد العمل، وتتم تحت
التهـديـد بـأي عقـاب، والتي لا يكون العـامـل قـد أداهـا بمحض اختيـاره؛ ويســتثنى من ذلـك أي عمـل، أو خـدمـة تفرضــهـا حـالات الطوارئ، أو القوة
القاهرة؛ مثل حالات الحروب، أو الحرائق، أو الفيضـانات، أو المجاعات، أو الزلازل، أو الأمراض الوبائية العنيفة، أو غزوات الحيوانات، أو الحشـرات، أو
الآفات النباتية، وغير ذلك
2. تطبق الفقرة الخامسة على العاملين في القطاع الخاص والقطاع غير الربحي.

Article: 31
Page: 22 - 22
Text:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، م

In [107]:
article_31 = next(
    article
    for article in executive_articles
    if article["article"] == "31"
)

marker = "المادية الحادية والستون بعد المائة:"

before, after = article_31["text"].split(marker, 1)

article_31["text"] = before.strip()

article_61 = {
    "section": "main_regulations",
    "article": "61",
    "text": after.strip(),
    "page_start": 22,
    "page_end": 22,
    "article_type": "executive_regulation",
    "related_law_articles": ["الحادية والستون"]
}

executive_articles.append(article_61)

print("Article 31:")
print(article_31["text"])

print("\n" + "=" * 60)

print("Article 61:")
print(article_61["text"])

Article 31:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بالوقت الذي تحدده لفترة أو فترات تلك الاسـتراحة،
وما يطرأ على ذ

In [108]:
print("Labor articles:", len(labor_articles))
print("Executive articles:", len(executive_articles))
print("Workplace articles:", len(workplace_articles))

print("\nExecutive article 31:")
print(next(a for a in executive_articles if a["article"] == "31")["text"][-200:])

print("\nExecutive article 61:")
print(next(a for a in executive_articles if a["article"] == "61")["text"])

Labor articles: 45
Executive articles: 46
Workplace articles: 71

Executive article 31:
دة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بالوقت الذي تحدده لفترة أو فترات تلك الاسـتراحة،
وما يطرأ على ذلك الوقت من تعديل.

Executive article 61:
لا يجوز تشـغيل الأحداث في الأعمال الخطرة أو الصـناعات الضـارة، أو في المهن والأعمال التي يحتمل أن تعرض صـحتهم أو سـلامتهم أو
أخلاقهم للخطر، بسبب طبيعتها أو الظروف التي تؤدى فيها. ويحدد الوزير بقرار منه الأعمال والصناعات والمهن المشار إليها.


In [109]:
all_articles = labor_articles + executive_articles + workplace_articles

rag_chunks = []

for article in all_articles:
    text = article["text"].strip()

    chunk = {
        "text": text,
        "metadata": {
            "section": article["section"],
            "article": article["article"],
            "article_type": article["article_type"],
            "page_start": article["page_start"],
            "page_end": article["page_end"]
        }
    }

    if "related_regulations" in article:
        chunk["metadata"]["related_regulations"] = article["related_regulations"]

    if "related_law_articles" in article:
        chunk["metadata"]["related_law_articles"] = article["related_law_articles"]

    rag_chunks.append(chunk)

print("Total RAG chunks:", len(rag_chunks))

from collections import Counter
print(Counter(
    chunk["metadata"]["article_type"]
    for chunk in rag_chunks
))

Total RAG chunks: 162
Counter({'workplace_regulation': 71, 'executive_regulation': 46, 'labor_law': 45})


In [110]:
for i, chunk in enumerate(rag_chunks):
    metadata = chunk["metadata"]

    if (
        metadata["article_type"] == "executive_regulation"
        and metadata["article"] in ["31", "61"]
    ):
        print("Index:", i)
        print("Article:", metadata["article"])
        print("Page:", metadata["page_start"])
        print("Text:")
        print(chunk["text"])
        print("\n" + "=" * 70)

Index: 80
Article: 31
Page: 22
Text:
في تنفيذ أحكام )المادة الرابعة والخمسون بعد المائة( من النظام:
1.
يحق للمرأة العاملة الحصـول على فترة، أو فترات اسـتراحة بقصـد إرضـاع مولودها، مدفوعة الأجر لا تزيد في مجموعها عن السـاعة في
اليوم الواحد؛ وتحسـب من سـاعات العمل الفعلية، وذلك لمدة أربعة وعشـرين شـهرًا من تاريخ الوضـع، سـواء أكانت الرضـاعة طبيعية، أو
غير ذلك.
2.
لا تعتبر فترة الاستراحة بقصد إرضاع مولودها، من ضمن فترات الراحة )الرسمية( المخصصة لجميع العاملين.
3.
تحدد فترة، أو فترات الاستراحة بقصد الإرضاع بحسب توفر الحضانة داخل المنشأة من عدمها، وفق التالي:
أ. إذا لم تتوفر حضـــانـة داخـل المنشـــأة: يحق للمرأة العـاملـة اختيـار فترة ســـاعـة الرضـــاعـة؛ إمـا أن تكون بـدايـة الـدوام، أو نهـايتـه؛ بمـا
يتناسب مع مصلحة العمل.
ب. إذا توفرت حضـانة داخل المنشـأة: يحق للمرأة العاملة اختيار فترة أو فترات اسـتراحة يكون إجمالي مدتها سـاعة واحدة للرضـاعة؛
بما يتناسب مع مصلحة العمل.
4.
يجب على المرأة العاملة بعد عودتها من إجازة الوضـع إشـعار صـاحب العمل كتابةً بالوقت الذي تحدده لفترة أو فترات تلك ا

In [111]:
# إعادة بناء embeddings من البيانات المصححة

rag_texts = [chunk["text"] for chunk in rag_chunks]

rag_embeddings = embedding_model.encode(
    rag_texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

rag_embeddings = np.array(rag_embeddings, dtype="float32")

print("Embeddings shape:", rag_embeddings.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings shape: (162, 768)


In [112]:
# إعادة بناء FAISS index من الـembeddings المصححة

embedding_dim = rag_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(rag_embeddings)

print("FAISS dimension:", faiss_index.d)
print("FAISS vectors:", faiss_index.ntotal)

FAISS dimension: 768
FAISS vectors: 162


In [113]:
# حفظ النسخ المصححة

with open(PROCESSED_DIR / "rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump(rag_chunks, f, ensure_ascii=False, indent=2)

np.save(
    PROCESSED_DIR / "rag_embeddings.npy",
    rag_embeddings
)

faiss.write_index(
    faiss_index,
    str(PROCESSED_DIR / "faiss.index")
)

print("Saved successfully:")
print("-", PROCESSED_DIR / "rag_chunks.json")
print("-", PROCESSED_DIR / "rag_embeddings.npy")
print("-", PROCESSED_DIR / "faiss.index")

Saved successfully:
- C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\rag_chunks.json
- C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\rag_embeddings.npy
- C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed\faiss.index


In [114]:
for chunk in rag_chunks:
    chunk["normalized_text"] = normalize_arabic(chunk["text"])

print("normalized_text added to all chunks.")

# تأكد
print("Total chunks:", len(rag_chunks))
print("Example:", rag_chunks[0]["normalized_text"][:100])

normalized_text added to all chunks.
Total chunks: 162
Example: تسري علي العامل العرضي والموسمي والموقت الاحكام الخاصة بالواجبات وقواعد التاديب، والحد الاقصي لساعات


In [115]:
test_question = "هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟"

results = hybrid_search(test_question, k=5)

for rank, result in enumerate(results, start=1):
    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print(f"--- Rank {rank} ---")
    print("Hybrid score:", round(result["hybrid_score"], 4))
    print("Semantic score:", round(result["semantic_score"], 4))
    print("Keyword score:", round(result["keyword_score"], 4))
    print("Type:", metadata["article_type"])
    print("Article:", metadata["article"])
    print("Page:", metadata["page_start"])
    print("Matched words:", result["matched_words"])
    print("Text:", chunk["text"][:500])
    print()

--- Rank 1 ---
Hybrid score: 0.6992
Semantic score: 0.7539
Keyword score: 0.5714
Type: executive_regulation
Article: 61
Page: 22
Matched words: ['تشغيل', 'الاحداث', 'الاعمال', 'يجوز']
Text: لا يجوز تشـغيل الأحداث في الأعمال الخطرة أو الصـناعات الضـارة، أو في المهن والأعمال التي يحتمل أن تعرض صـحتهم أو سـلامتهم أو
أخلاقهم للخطر، بسبب طبيعتها أو الظروف التي تؤدى فيها. ويحدد الوزير بقرار منه الأعمال والصناعات والمهن المشار إليها.

--- Rank 2 ---
Hybrid score: 0.6729
Semantic score: 0.6551
Keyword score: 0.7143
Type: labor_law
Article: السادسة بعد المائة
Page: 16
Matched words: ['تشغيل', 'العمل', 'لصاحب', 'الاعمال', 'يجوز']
Text: يجوز لصــاحب العمل عدم التقيد بأحكام المواد الثامنة والتســعين والأولى بعد المائة والفقرة )1( من المادة الرابعة بعد المائة من هذا
النظام في الحالات الآتية:
1. أعمال الجرد السـنوي، وإعداد الميزانية، والتص
ـفية، وقفل الحسـابات، والاسـتعداد للبيع بأثمان مخفض
ـة، والاسـتعداد للمواسـم، بشـرط ألا
يزيد عدد الأيام التي يشتغل فيها العمال على ثلاثين يومًا في السنة.
2. إذا ك

In [116]:
for i, question in enumerate(evaluation_questions, start=1):
    results = hybrid_search(question, k=1)

    result = results[0]
    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print(f"{i}. {question}")
    print(
        f"   → {metadata['article_type']} | "
        f"المادة {metadata['article']} | "
        f"الصفحة {metadata['page_start']} | "
        f"score={result['hybrid_score']:.4f}"
    )
    print()

1. كم مدة إجازة الوضع للمرأة العاملة؟
   → workplace_regulation | المادة 50 | الصفحة 36 | score=0.8265

2. هل إجازة الوضع للمرأة العاملة بأجر كامل؟
   → workplace_regulation | المادة 50 | الصفحة 36 | score=0.8248

3. ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟
   → workplace_regulation | المادة 50 | الصفحة 36 | score=0.5240

4. كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟
   → executive_regulation | المادة 22 | الصفحة 17 | score=0.7004

5. ما الحد الأقصى لمدة فترة التجربة؟
   → workplace_regulation | المادة 8 | الصفحة 30 | score=0.5028

6. ما هي حقوق المرأة العاملة في فترة الرضاعة؟
   → executive_regulation | المادة 31 | الصفحة 22 | score=0.7104

7. هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟
   → executive_regulation | المادة 61 | الصفحة 22 | score=0.6992

8. ما المقصود بالشخص ذي الإعاقة في اللائحة؟
   → executive_regulation | المادة 9 | الصفحة 6 | score=0.7080

9. ما هي مدة إجازة الوضع للمرأة العاملة إذا تأخر 

In [117]:
result = ask_question_hybrid(
    "هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟",
    k=3
)

print("ANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(
        f"Rank {source['rank']} | "
        f"{source['article_type']} | "
        f"المادة {source['article']} | "
        f"الصفحة {source['page']} | "
        f"score={source['score']:.4f}"
    )

ANSWER:
لا يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة أو الصناعات الضارة، أو في المهن والأعمال التي يحتمل أن تعرض صحتهم أو سلامتهم أو أخلاقهم للخطر (المادة 61).

SOURCES:
Rank 1 | executive_regulation | المادة 61 | الصفحة 22 | score=0.6992
Rank 2 | labor_law | المادة السادسة بعد المائة | الصفحة 16 | score=0.6729
Rank 3 | labor_law | المادة التاسعة والثلاثون | الصفحة 7 | score=0.6361


In [118]:
RETRIEVAL_THRESHOLD = 0.50


def ask_question_hybrid(question, k=3, threshold=RETRIEVAL_THRESHOLD):

    results = hybrid_search(question, k=k)

    best_score = results[0]["hybrid_score"]

    sources = []

    for rank, result in enumerate(results, start=1):
        chunk = rag_chunks[result["idx"]]
        metadata = chunk["metadata"]

        sources.append({
            "rank": rank,
            "score": result["hybrid_score"],
            "article_type": metadata["article_type"],
            "article": metadata["article"],
            "page": metadata["page_start"]
        })

    # Reject weak retrieval
    if best_score < threshold:
        return {
            "answer": "لا يحتوي السياق المسترجع على معلومات كافية للإجابة.",
            "sources": sources,
            "rejected": True
        }

    # Build context
    context_parts = []

    for rank, result in enumerate(results, start=1):
        chunk = rag_chunks[result["idx"]]
        metadata = chunk["metadata"]

        context_parts.append(
            f"""المصدر {rank}:
نوع المصدر: {metadata["article_type"]}
المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص:
{chunk["text"]}
"""
        )

    context = "\n\n".join(context_parts)

    user_prompt = f"""السؤال:
{question}

السياق القانوني المسترجع:
{context}

أجب عن السؤال بناءً على السياق أعلاه.
أجب باللغة العربية فقط وباختصار."""

    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            "temperature": 0.0,
            "max_tokens": 300
        },
        timeout=120
    )

    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"]

    return {
        "answer": answer,
        "sources": sources,
        "rejected": False
    }


# ============================================================
# Run tests
# ============================================================

test_questions = [
    "هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟",
    "ما هو الحد الأدنى لرأس مال الشركة؟",
    "كم مدة إجازة الوضع للمرأة العاملة؟"
]

for i, question in enumerate(test_questions, start=1):

    result = ask_question_hybrid(question)

    print("=" * 70)
    print(f"TEST {i}")
    print("QUESTION:", question)
    print("=" * 70)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print(
            f"Rank {source['rank']} | "
            f"{source['article_type']} | "
            f"المادة {source['article']} | "
            f"الصفحة {source['page']} | "
            f"score={source['score']:.4f}"
        )

    print("\nRejected:", result["rejected"])
    print()

TEST 1
QUESTION: هل يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة؟

ANSWER:
لا يجوز لصاحب العمل تشغيل الأحداث في الأعمال الخطرة أو الصناعات الضارة، أو في المهن والأعمال التي يحتمل أن تعرض صحتهم أو سلامتهم أو أخلاقهم للخطر (المادة 61).

SOURCES:
Rank 1 | executive_regulation | المادة 61 | الصفحة 22 | score=0.6992
Rank 2 | labor_law | المادة السادسة بعد المائة | الصفحة 16 | score=0.6729
Rank 3 | labor_law | المادة التاسعة والثلاثون | الصفحة 7 | score=0.6361

Rejected: False

TEST 2
QUESTION: ما هو الحد الأدنى لرأس مال الشركة؟

ANSWER:
لا يحتوي السياق المسترجع على معلومات كافية للإجابة.

SOURCES:
Rank 1 | executive_regulation | المادة 29 | الصفحة 21 | score=0.4019
Rank 2 | workplace_regulation | المادة 21 | الصفحة 31 | score=0.3886
Rank 3 | labor_law | المادة السادسة بعد المائة | الصفحة 16 | score=0.3397

Rejected: True

TEST 3
QUESTION: كم مدة إجازة الوضع للمرأة العاملة؟

ANSWER:
مدة إجازة الوضع للمرأة العاملة هي اثنتا عشرة أسبوعًا (المادة 50).

SOURCES:
Rank 1 | workplace_regulation 